In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev_ua5816bd", "Catalog Name")
dbutils.widgets.text("bronze_schema", "oles0305_bronze", "Bronze Schema")
dbutils.widgets.text("silver_schema", "oles0305_silver", "Silver Schema")

CATALOG_NAME = dbutils.widgets.get("catalog_name")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

BRONZE = f"{CATALOG_NAME}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG_NAME}.{SILVER_SCHEMA}"

### SCD Type 2 in silver `drivers` table

In [0]:
# Ingest new record for Lewis Hamilton (driverId=1) in bronze drivers table with changing his number from 44 to 99
spark.sql(f"""
    INSERT INTO {BRONZE}.drivers
    SELECT 
        1 AS driverId,
        'hamilton' AS driverRef,
        99 AS number, -- new number
        'HAM' AS code,
        'Lewis' AS forename,
        'Hamilton' AS surname,
        CAST('1985-01-07' AS DATE) AS dob,
        'British' AS nationality,
        'http://example.com' AS url,
        'manual_test' AS source_filename,
        current_timestamp() AS ingestion_timestamp,
        current_date() AS load_date
""")

In [0]:
dbutils.notebook.run("01_silver_drivers", timeout_seconds=120, arguments={
    "catalog_name": CATALOG_NAME,
    "bronze_schema": BRONZE_SCHEMA,
    "silver_schema": SILVER_SCHEMA
})

Result:
![](screenshots/scd_2_result.png)

### Schema Enforcement test

In [0]:
df_invalid = spark.sql(f"""
    SELECT 999998 as resultId, 
        1 as raceId, 
        1 as driverId, 
        1 as constructorId,
        1 as grid, 
        1 as position, 
        CAST(25.0 AS DOUBLE) as points, 
        50 as laps, 
        5000000 as milliseconds,
        1 as statusId, 
        '1:20.000' as time,
        current_timestamp() as bronze_ingested_at,
        current_timestamp() as silver_processed_at,
        'Rain' as weather_condition  -- extra column
""")

Result:
![](screenshots/schema_enforcement_result.png)

### Schema Evolution test (enabling AutoMerge)

In [0]:
df_evolved = spark.sql(f"""
    SELECT 999999 as resultId, 
        1 AS raceId, 
        1 AS driverId, 
        1 AS constructorId,
        CAST(20 AS BIGINT) AS grid, -- widening (INT -> BIGINT)
        1 AS position, 
        CAST(25.0 AS DOUBLE) AS points, 
        50 AS laps, 
        5000000 AS milliseconds, 
        1 AS statusId, 
        '1:20.000' AS time,
        current_timestamp() as bronze_ingested_at,
        current_timestamp() as silver_processed_at,
        'Dry' as weather_condition  -- extra column
""")

In [0]:
(
    df_evolved.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"{SILVER}.results")
)

Result:
![](screenshots/schema_evolution_result.png)

### Delta Column Mapping test

In [0]:
spark.sql(f"ALTER TABLE {SILVER}.drivers RENAME COLUMN dob TO date_of_birth")
spark.sql(f"ALTER TABLE {SILVER}.drivers DROP COLUMN url")

Result:
![](screenshots/column_mapping_result.png)

---

### Discuss data contracts as the controlled alternative to silent evolution. 
**Silent evolution** (features like `mergeSchema` or `autoMerge`) is convenient during development, but it can introduce some risks in production, as unexpected change in data structure even due to a small mistake in data source can lead to breaking downstream consumers if they expect fixed structures. Also silent evolution accepts unintendent changes in schema without human validation.

**Data contracts** allow us to specify expectations on schema, data quality and other things between Data Producer and Consumer. Validation is conducted before writing, so if there are any invalid records, they don't change our data structure but are rejected or redirected to a quarantine table.

### OPTIMIZE, VACUUM, and Liquid Clustering (the current recommendation) versus ZORDER and classic partitioning — understand the trade-offs.

- **OPTIMIZE** solves the small files problem. It comapcts small Parquet files into more optimal larger Parquet files and adds them to the Delta transactional log, and marks old files as tombstoned.

- **VACUUM** removes tombstoned files that are older than the specified retention time.

---

- **Classic Partitioning** (`PARTITION BY`) physically organizes files into folders by the specified partition column. Its trade-offs are small file problem, data skew, and changing partition keys requires rewriting all existing data.

- **ZORDER** organizes related information in the same set of files so that rows with similar values for specified columns are stored close together. Spark uses the minimum and maximum statistics of these files to completely skip reading unnecessary data when executing a query. But writes don't automatically add to the Z-curve so it requires heavy manual rewrites.

- **Liquid Clustering** automatically groups data into files based on specified clustering keys to maximize efficient data skipping during queries. Organizes newly incoming data incrementally without causing heavy writes. Also it allows you to change your clustering columns using SQL ALTER TABLE commands without needing to rewrite existing historical data.